In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/Documentation_template.md
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/README.md
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/utils/validate_submission.py
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source2.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source3.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source1.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train/train_ground_truth.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train/train_source3.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train/train_source2.tsv
/kaggl

# Matching Model

This notebook trains a pairwise entity-matching model using blocked candidate pairs and training ground truth.

The model learns whether a candidate source-2 or source-3 record matches a source-1 business and selects a precision-oriented threshold using the F0.5 evaluation metric.

In [2]:
# v5 - Matching Model

import os
import gc
import re
import unicodedata
import random
import warnings
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score,
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

DATA_ROOT = (
    "/kaggle/input/datasets/"
    "archangel26/infinity8-amazon-ml-challenge/"
    "student_resource"
)

TRAIN_DIR = os.path.join(
    DATA_ROOT,
    "dataset",
    "train",
)

MODEL_DIR = "/kaggle/working/matching_model"

os.makedirs(
    MODEL_DIR,
    exist_ok=True,
)

print("Environment ready.")
print("TRAIN_DIR:", TRAIN_DIR)
print("MODEL_DIR:", MODEL_DIR)

Environment ready.
TRAIN_DIR: /kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train
MODEL_DIR: /kaggle/working/matching_model


## 1. Configuration

Define the training sample size, blocking limits, negative sampling limit, and XGBoost parameters.

The configuration is intentionally memory-conscious because the complete training sources contain more than twelve million records.

In [3]:
TRAIN_S1_SAMPLE = 10_000

MAX_NEGATIVES_PER_S1 = 20

MAX_BLOCK_SIZE = 500

VALID_SIZE = 0.20

TRAIN_BLOCKING_RULES = [
    "name_country_key",
    "name_first_country_key",
    "name_prefix3_country_key",
    "name_prefix4_country_key",
    "address_country_key",
]

XGB_PARAMS = {
    "n_estimators": 500,
    "max_depth": 7,
    "learning_rate": 0.08,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": -1,
}

print("S1 training sample:", TRAIN_S1_SAMPLE)
print("Maximum negatives per S1:", MAX_NEGATIVES_PER_S1)
print("Maximum block size:", MAX_BLOCK_SIZE)
print("Training rules:", TRAIN_BLOCKING_RULES)

S1 training sample: 10000
Maximum negatives per S1: 20
Maximum block size: 500
Training rules: ['name_country_key', 'name_first_country_key', 'name_prefix3_country_key', 'name_prefix4_country_key', 'address_country_key']


## 2. Text Normalization

Recreate the lightweight Unicode normalization used during preprocessing so the matching model uses the same normalized representation as the blocking pipeline.

The normalization preserves Unicode letters, numbers, and combining marks while removing punctuation noise.

In [4]:
def normalize_unicode_text(text):
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    text = text.replace(
        "&",
        " and ",
    )

    cleaned = []

    for char in text:
        category = unicodedata.category(char)

        if char.isspace():
            cleaned.append(" ")

        elif category[0] in (
            "L",
            "N",
            "M",
        ):
            cleaned.append(char)

        else:
            cleaned.append(" ")

    text = "".join(cleaned)

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def add_normalized_fields(df):

    df["name_norm"] = (
        df["business_name"]
        .fillna("")
        .map(normalize_unicode_text)
    )

    df["address_norm"] = (
        df["business_address"]
        .fillna("")
        .map(normalize_unicode_text)
    )

    df["country_key"] = (
        df["country"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df["name_first_token"] = (
        df["name_norm"]
        .str.split()
        .str[0]
        .fillna("")
    )

    df["name_last_token"] = (
        df["name_norm"]
        .str.split()
        .str[-1]
        .fillna("")
    )

    compact_name = (
        df["name_norm"]
        .str.replace(
            " ",
            "",
            regex=False,
        )
    )

    df["name_country_key"] = (
        df["name_norm"]
        + "\t"
        + df["country_key"]
    )

    df["name_first_country_key"] = (
        df["name_first_token"]
        + "\t"
        + df["country_key"]
    )

    df["name_prefix3_country_key"] = (
        compact_name.str[:3]
        + "\t"
        + df["country_key"]
    )

    df["name_prefix4_country_key"] = (
        compact_name.str[:4]
        + "\t"
        + df["country_key"]
    )

    df["address_country_key"] = (
        df["address_norm"]
        + "\t"
        + df["country_key"]
    )


def cleanup_raw_columns(df):

    drop_cols = [
        "business_name",
        "business_address",
        "country",
    ]

    existing = [
        c for c in drop_cols
        if c in df.columns
    ]

    if existing:
        df.drop(
            columns=existing,
            inplace=True,
        )

## 3. Load Training Data

Load the three training sources and the ground-truth mapping using only the columns required for matching.

No test data is loaded or processed in this notebook.

In [5]:
USECOLS = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

train_s1 = pd.read_csv(
    os.path.join(
        TRAIN_DIR,
        "train_source1.tsv",
    ),
    sep="\t",
    dtype=str,
    usecols=USECOLS,
)

train_s2 = pd.read_csv(
    os.path.join(
        TRAIN_DIR,
        "train_source2.tsv",
    ),
    sep="\t",
    dtype=str,
    usecols=USECOLS,
)

train_s3 = pd.read_csv(
    os.path.join(
        TRAIN_DIR,
        "train_source3.tsv",
    ),
    sep="\t",
    dtype=str,
    usecols=USECOLS,
)

train_gt = pd.read_csv(
    os.path.join(
        TRAIN_DIR,
        "train_ground_truth.tsv",
    ),
    sep="\t",
    dtype=str,
)

print("S1:", train_s1.shape)
print("S2:", train_s2.shape)
print("S3:", train_s3.shape)
print("GT:", train_gt.shape)

S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)
GT: (2206821, 2)


## 4. Prepare Training Tables

Assign compact integer row identifiers and create the normalized fields required for blocking and pairwise feature generation.

The original text columns are removed afterward to reduce memory usage.

In [6]:
train_s1["s1_idx"] = np.arange(
    len(train_s1),
    dtype=np.int32,
)

train_s2["s2_idx"] = np.arange(
    len(train_s2),
    dtype=np.int32,
)

train_s3["s3_idx"] = np.arange(
    len(train_s3),
    dtype=np.int32,
)

add_normalized_fields(train_s1)
add_normalized_fields(train_s2)
add_normalized_fields(train_s3)

cleanup_raw_columns(train_s1)
cleanup_raw_columns(train_s2)
cleanup_raw_columns(train_s3)

gc.collect()

print("Training tables prepared.")

print(
    "S1 columns:",
    train_s1.columns.tolist(),
)

print(
    "S2 columns:",
    train_s2.columns.tolist(),
)

print(
    "S3 columns:",
    train_s3.columns.tolist(),
)

Training tables prepared.
S1 columns: ['entity_id', 's1_idx', 'name_norm', 'address_norm', 'country_key', 'name_first_token', 'name_last_token', 'name_country_key', 'name_first_country_key', 'name_prefix3_country_key', 'name_prefix4_country_key', 'address_country_key']
S2 columns: ['entity_id', 's2_idx', 'name_norm', 'address_norm', 'country_key', 'name_first_token', 'name_last_token', 'name_country_key', 'name_first_country_key', 'name_prefix3_country_key', 'name_prefix4_country_key', 'address_country_key']
S3 columns: ['entity_id', 's3_idx', 'name_norm', 'address_norm', 'country_key', 'name_first_token', 'name_last_token', 'name_country_key', 'name_first_country_key', 'name_prefix3_country_key', 'name_prefix4_country_key', 'address_country_key']


## 5. Build Ground-Truth Mappings

Convert the ground-truth entity IDs into compact source-aware row-index pairs.

Source identity is retained so an S2 row and an S3 row with the same numeric index are never confused.

In [7]:
s1_id_to_idx = dict(
    zip(
        train_s1["entity_id"],
        train_s1["s1_idx"],
    )
)

s2_id_to_idx = dict(
    zip(
        train_s2["entity_id"],
        train_s2["s2_idx"],
    )
)

s3_id_to_idx = dict(
    zip(
        train_s3["entity_id"],
        train_s3["s3_idx"],
    )
)

true_pairs_by_s1 = {}

for row in train_gt.itertuples(
    index=False
):

    s1_id = row.source1_entity_id
    matched_ids = row.matched_entity_ids

    s1_idx = s1_id_to_idx.get(
        s1_id
    )

    if s1_idx is None:
        continue

    pair_set = set()

    if pd.notna(matched_ids):

        matched_ids = str(
            matched_ids
        ).strip()

        if matched_ids:

            for matched_id in matched_ids.split(","):

                matched_id = matched_id.strip()

                if not matched_id:
                    continue

                if matched_id in s2_id_to_idx:

                    pair_set.add(
                        (
                            "s2",
                            int(
                                s2_id_to_idx[
                                    matched_id
                                ]
                            ),
                        )
                    )

                if matched_id in s3_id_to_idx:

                    pair_set.add(
                        (
                            "s3",
                            int(
                                s3_id_to_idx[
                                    matched_id
                                ]
                            ),
                        )
                    )

    true_pairs_by_s1[
        int(s1_idx)
    ] = pair_set

print(
    "Ground-truth S1 entities:",
    f"{len(true_pairs_by_s1):,}",
)

print(
    "Total true pairs:",
    f"{sum(len(v) for v in true_pairs_by_s1.values()):,}",
)

Ground-truth S1 entities: 2,206,821
Total true pairs: 7,638,365


## 7. Generate Training Candidates

Generate candidate pairs using the same blocking philosophy as the dedicated blocking notebook.

Blocks larger than the configured limit are skipped because extremely common keys produce low-quality candidate explosions.

In [19]:
def generate_training_candidates(
    s1_sample,
    source_df,
    source_name,
    rules,
):

    source_idx_col = (
        "s2_idx"
        if source_name == "s2"
        else "s3_idx"
    )

    candidate_parts = []

    for rule in rules:

        print(
            f"{source_name} | {rule}"
        )

        counts = (
            source_df[rule]
            .value_counts()
        )

        valid_keys = counts[
            (counts <= MAX_BLOCK_SIZE)
            & (counts.index != "")
        ].index

        if len(valid_keys) == 0:
            del counts
            gc.collect()
            continue

        right = source_df[
            source_df[rule].isin(
                valid_keys
            )
        ][
            [
                rule,
                source_idx_col,
            ]
        ]

        left = s1_sample[
            [
                "s1_idx",
                rule,
            ]
        ]

        left = left[
            left[rule].ne("")
        ]

        if left.empty or right.empty:
            del counts
            del valid_keys
            del right
            del left
            gc.collect()
            continue

        merged = left.merge(
            right,
            on=rule,
            how="inner",
            sort=False,
        )

        if not merged.empty:

            merged = merged[
                [
                    "s1_idx",
                    source_idx_col,
                ]
            ].rename(
                columns={
                    source_idx_col:
                    "matched_idx"
                }
            )

            merged["source"] = (
                source_name
            )

            candidate_parts.append(
                merged
            )

        del counts
        del valid_keys
        del right
        del left
        del merged

        gc.collect()

    if not candidate_parts:

        return pd.DataFrame(
            columns=[
                "s1_idx",
                "matched_idx",
                "source",
            ]
        )

    candidates = pd.concat(
        candidate_parts,
        ignore_index=True,
    )

    del candidate_parts

    candidates.drop_duplicates(
        subset=[
            "s1_idx",
            "matched_idx",
            "source",
        ],
        inplace=True,
    )

    gc.collect()

    return candidates


sample_size = min(
    TRAIN_S1_SAMPLE,
    len(train_s1),
)

sampled_s1_idx = (
    train_s1["s1_idx"]
    .sample(
        n=sample_size,
        random_state=SEED,
    )
    .to_numpy()
)

sampled_s1_idx = np.sort(
    sampled_s1_idx
)

s1_sample = train_s1[
    train_s1["s1_idx"].isin(
        sampled_s1_idx
    )
].copy()

print(
    "Selected S1 entities:",
    f"{len(s1_sample):,}",
)

print(
    "Expected:",
    f"{sample_size:,}",
)

Selected S1 entities: 10,000
Expected: 10,000


Selected S1 entities: 10,000
Expected: 10,000


In [17]:
print("s1_sample exists:", "s1_sample" in globals())

if "s1_sample" in globals():
    print("s1_sample shape:", s1_sample.shape)
    print("s1_sample columns:", list(s1_sample.columns))

s1_sample exists: True
s1_sample shape: (10000, 12)
s1_sample columns: ['entity_id', 's1_idx', 'name_norm', 'address_norm', 'country_key', 'name_first_token', 'name_last_token', 'name_country_key', 'name_first_country_key', 'name_prefix3_country_key', 'name_prefix4_country_key', 'address_country_key']


## 8. Create Source-2 and Source-3 Candidates

Generate blocked candidate pairs separately for source 2 and source 3.

Candidates from different blocking rules are deduplicated because the downstream model only needs one copy of each pair.

In [20]:
print("Generating source-2 candidates...")

pairs_s2_all = generate_training_candidates(
    s1_sample=s1_sample,
    source_df=train_s2,
    source_name="s2",
    rules=TRAIN_BLOCKING_RULES,
)

print(
    "Source-2 candidates:",
    f"{len(pairs_s2_all):,}",
)

gc.collect()

print("\nGenerating source-3 candidates...")

pairs_s3_all = generate_training_candidates(
    s1_sample=s1_sample,
    source_df=train_s3,
    source_name="s3",
    rules=TRAIN_BLOCKING_RULES,
)

print(
    "Source-3 candidates:",
    f"{len(pairs_s3_all):,}",
)

gc.collect()

print(
    "\nCombined candidates:",
    f"{len(pairs_s2_all) + len(pairs_s3_all):,}",
)

Generating source-2 candidates...
s2 | name_country_key
s2 | name_first_country_key
s2 | name_prefix3_country_key
s2 | name_prefix4_country_key
s2 | address_country_key
Source-2 candidates: 979,519

Generating source-3 candidates...
s3 | name_country_key
s3 | name_first_country_key
s3 | name_prefix3_country_key
s3 | name_prefix4_country_key
s3 | address_country_key
Source-3 candidates: 984,592

Combined candidates: 1,964,111


## 9. Label Candidate Pairs

Assign a positive label when a blocked candidate appears in the ground truth.

Pairs absent from the ground truth are treated as negative candidates for model training.

In [21]:
def label_candidates(
    candidates,
    true_pairs_by_s1,
):

    labels = np.zeros(
        len(candidates),
        dtype=np.int8,
    )

    for i, row in enumerate(
        candidates.itertuples(
            index=False
        )
    ):

        true_set = (
            true_pairs_by_s1.get(
                int(row.s1_idx),
                set(),
            )
        )

        if (
            row.source,
            int(row.matched_idx),
        ) in true_set:

            labels[i] = 1

    result = candidates.copy()

    result["label"] = labels

    return result


pairs_s2_all = label_candidates(
    pairs_s2_all,
    true_pairs_by_s1,
)

pairs_s3_all = label_candidates(
    pairs_s3_all,
    true_pairs_by_s1,
)

print(
    "S2 positives:",
    f"{int(pairs_s2_all['label'].sum()):,}",
)

print(
    "S3 positives:",
    f"{int(pairs_s3_all['label'].sum()):,}",
)

S2 positives: 9,567
S3 positives: 9,800


## 10. Balance Candidate Pairs

Keep all discovered positive pairs while limiting the number of negative pairs per source-1 entity.

This reduces the extreme class imbalance and prevents unnecessary memory usage during feature construction.

In [22]:
def sample_pair_dataset(
    candidates,
    max_negatives_per_s1,
    seed,
):

    positives = candidates[
        candidates["label"] == 1
    ].copy()

    negatives = candidates[
        candidates["label"] == 0
    ].copy()

    if negatives.empty:

        return candidates.copy()

    negative_parts = []

    for s1_idx, group in negatives.groupby(
        "s1_idx",
        sort=False,
    ):

        n = min(
            len(group),
            max_negatives_per_s1,
        )

        negative_parts.append(
            group.sample(
                n=n,
                random_state=seed,
            )
        )

    negatives_sampled = pd.concat(
        negative_parts,
        ignore_index=True,
    )

    result = pd.concat(
        [
            positives,
            negatives_sampled,
        ],
        ignore_index=True,
    )

    result.drop_duplicates(
        subset=[
            "s1_idx",
            "matched_idx",
            "source",
        ],
        inplace=True,
    )

    return result


pairs_s2 = sample_pair_dataset(
    pairs_s2_all,
    MAX_NEGATIVES_PER_S1,
    SEED,
)

pairs_s3 = sample_pair_dataset(
    pairs_s3_all,
    MAX_NEGATIVES_PER_S1,
    SEED,
)

del pairs_s2_all
del pairs_s3_all

gc.collect()

print(
    "Final S2 pairs:",
    f"{len(pairs_s2):,}",
)

print(
    "Final S3 pairs:",
    f"{len(pairs_s3):,}",
)

print(
    "S2 positives:",
    f"{int(pairs_s2['label'].sum()):,}",
)

print(
    "S3 positives:",
    f"{int(pairs_s3['label'].sum()):,}",
)

Final S2 pairs: 128,198
Final S3 pairs: 130,954
S2 positives: 9,567
S3 positives: 9,800


## 11. String Similarity Functions

Define lightweight string similarity functions using Python's standard library.

No additional fuzzy-matching package is required, keeping the notebook compatible with the Kaggle environment.

In [24]:
def safe_ratio(a, b):

    if not a or not b:
        return 0.0

    return SequenceMatcher(
        None,
        a,
        b,
    ).ratio()


def safe_token_sort(a, b):

    if not a or not b:
        return 0.0

    a_sorted = " ".join(
        sorted(a.split())
    )

    b_sorted = " ".join(
        sorted(b.split())
    )

    return safe_ratio(
        a_sorted,
        b_sorted,
    )


def safe_token_set(a, b):

    if not a or not b:
        return 0.0

    a_tokens = set(
        a.split()
    )

    b_tokens = set(
        b.split()
    )

    if not a_tokens or not b_tokens:
        return 0.0

    intersection = (
        a_tokens & b_tokens
    )

    union = (
        a_tokens | b_tokens
    )

    return (
        len(intersection)
        / len(union)
    )


def safe_partial_ratio(a, b):

    if not a or not b:
        return 0.0

    # Use the shorter string as the query.
    if len(a) > len(b):
        a, b = b, a

    if len(a) == len(b):
        return safe_ratio(a, b)

    window = len(a)

    # For long strings, sample aligned windows rather than
    # performing an expensive exhaustive search.
    if len(b) - window > 100:

        positions = np.linspace(
            0,
            len(b) - window,
            num=20,
            dtype=int,
        )

    else:

        positions = range(
            len(b) - window + 1
        )

    best = 0.0

    for pos in positions:

        score = SequenceMatcher(
            None,
            a,
            b[
                pos:pos + window
            ],
        ).ratio()

        if score > best:
            best = score

    return best


def safe_wratio(a, b):

    if not a or not b:
        return 0.0

    return max(
        safe_ratio(a, b),
        safe_token_sort(a, b),
        safe_token_set(a, b),
    )


def length_ratio(a, b):

    la = len(a)
    lb = len(b)

    if la == 0 and lb == 0:
        return 1.0

    if la == 0 or lb == 0:
        return 0.0

    return min(
        la,
        lb,
    ) / max(
        la,
        lb,
    )


def token_overlap(a, b):

    ta = set(a.split())
    tb = set(b.split())

    if not ta or not tb:
        return 0.0

    return len(ta & tb) / len(
        ta | tb
    )


def digit_overlap(a, b):

    da = set(
        ch for ch in a
        if ch.isdigit()
    )

    db = set(
        ch for ch in b
        if ch.isdigit()
    )

    if not da and not db:
        return 1.0

    if not da or not db:
        return 0.0

    return len(da & db) / len(
        da | db
    )

## 12. Define Pair Features

Construct the numeric feature set used by XGBoost.

The features capture name similarity, address similarity, token overlap, digit overlap, exact matches, missingness, and source identity.

In [25]:
FEATURE_COLUMNS = [
    "name_ratio",
    "name_partial_ratio",
    "name_token_sort",
    "name_token_set",
    "name_wratio",
    "name_length_ratio",
    "name_token_overlap",
    "address_ratio",
    "address_partial_ratio",
    "address_token_sort",
    "address_token_set",
    "address_wratio",
    "address_length_ratio",
    "address_token_overlap",
    "digit_overlap",
    "country_exact",
    "name_exact",
    "address_exact",
    "name_missing",
    "address_missing",
    "is_source3",
]


def build_features(
    candidates,
    s1_df,
    source_df,
    source,
):

    if candidates.empty:

        return pd.DataFrame(
            columns=FEATURE_COLUMNS
        )

    source_idx_col = (
        "s2_idx"
        if source == "s2"
        else "s3_idx"
    )

    s1_lookup = (
        s1_df
        .set_index("s1_idx")
    )

    source_lookup = (
        source_df
        .set_index(source_idx_col)
    )

    left = s1_lookup.loc[
        candidates[
            "s1_idx"
        ].to_numpy()
    ]

    right = source_lookup.loc[
        candidates[
            "matched_idx"
        ].to_numpy()
    ]

    rows = []

    for i in range(
        len(candidates)
    ):

        name_a = left[
            "name_norm"
        ].iloc[i]

        name_b = right[
            "name_norm"
        ].iloc[i]

        addr_a = left[
            "address_norm"
        ].iloc[i]

        addr_b = right[
            "address_norm"
        ].iloc[i]

        country_a = left[
            "country_key"
        ].iloc[i]

        country_b = right[
            "country_key"
        ].iloc[i]

        rows.append(
            (
                safe_ratio(
                    name_a,
                    name_b,
                ),

                safe_partial_ratio(
                    name_a,
                    name_b,
                ),

                safe_token_sort(
                    name_a,
                    name_b,
                ),

                safe_token_set(
                    name_a,
                    name_b,
                ),

                safe_wratio(
                    name_a,
                    name_b,
                ),

                length_ratio(
                    name_a,
                    name_b,
                ),

                token_overlap(
                    name_a,
                    name_b,
                ),

                safe_ratio(
                    addr_a,
                    addr_b,
                ),

                safe_partial_ratio(
                    addr_a,
                    addr_b,
                ),

                safe_token_sort(
                    addr_a,
                    addr_b,
                ),

                safe_token_set(
                    addr_a,
                    addr_b,
                ),

                safe_wratio(
                    addr_a,
                    addr_b,
                ),

                length_ratio(
                    addr_a,
                    addr_b,
                ),

                token_overlap(
                    addr_a,
                    addr_b,
                ),

                digit_overlap(
                    addr_a,
                    addr_b,
                ),

                int(
                    bool(country_a)
                    and bool(country_b)
                    and country_a == country_b
                ),

                int(
                    bool(name_a)
                    and bool(name_b)
                    and name_a == name_b
                ),

                int(
                    bool(addr_a)
                    and bool(addr_b)
                    and addr_a == addr_b
                ),

                int(
                    not name_a
                    or not name_b
                ),

                int(
                    not addr_a
                    or not addr_b
                ),

                int(
                    source == "s3"
                ),
            )
        )

    return pd.DataFrame(
        rows,
        columns=FEATURE_COLUMNS,
        dtype=np.float32,
    )

## 13. Build Source-2 Features

Calculate pairwise features for the sampled source-2 candidates.

The resulting matrix contains only numeric values and can therefore be passed directly to XGBoost.

In [26]:
print(
    "Building source-2 features..."
)

X_s2 = build_features(
    candidates=pairs_s2,
    s1_df=train_s1,
    source_df=train_s2,
    source="s2",
)

print(
    "S2 feature matrix:",
    X_s2.shape,
)

gc.collect()

Building source-2 features...
S2 feature matrix: (128198, 21)


0

## 14. Build Source-3 Features

Calculate the same pairwise feature set for source-3 candidates.

Keeping source-2 and source-3 feature construction separate reduces peak memory usage.

In [27]:
print(
    "Building source-3 features..."
)

X_s3 = build_features(
    candidates=pairs_s3,
    s1_df=train_s1,
    source_df=train_s3,
    source="s3",
)

print(
    "S3 feature matrix:",
    X_s3.shape,
)

gc.collect()

Building source-3 features...
S3 feature matrix: (130954, 21)


0

## 15. Combine Training Features

Combine the source-2 and source-3 feature matrices while preserving exactly the same row order in the candidate metadata and labels.

This alignment is required for correct validation and entity-level evaluation.

In [28]:
X = pd.concat(
    [
        X_s2,
        X_s3,
    ],
    ignore_index=True,
)

y = pd.concat(
    [
        pairs_s2["label"],
        pairs_s3["label"],
    ],
    ignore_index=True,
).astype(
    np.int8
)

all_meta = pd.concat(
    [
        pairs_s2[
            [
                "s1_idx",
                "matched_idx",
                "source",
            ]
        ],
        pairs_s3[
            [
                "s1_idx",
                "matched_idx",
                "source",
            ]
        ],
    ],
    ignore_index=True,
)

del X_s2
del X_s3

gc.collect()

print(
    "Feature matrix:",
    X.shape,
)

print(
    "Metadata rows:",
    all_meta.shape,
)

print(
    "Labels:",
    y.shape,
)

print(
    "Positive pairs:",
    f"{int(y.sum()):,}",
)

print(
    "Negative pairs:",
    f"{int((y == 0).sum()):,}",
)

Feature matrix: (259152, 21)
Metadata rows: (259152, 3)
Labels: (259152,)
Positive pairs: 19,367
Negative pairs: 239,785


## 16. Split by Source-1 Entity

Create training and validation sets by source-1 entity rather than by individual pair.

This prevents candidate pairs belonging to the same source-1 business from leaking between training and validation.

In [29]:
unique_s1 = (
    all_meta["s1_idx"]
    .unique()
)

train_s1_ids, valid_s1_ids = (
    train_test_split(
        unique_s1,
        test_size=VALID_SIZE,
        random_state=SEED,
    )
)

train_mask = (
    all_meta["s1_idx"]
    .isin(train_s1_ids)
)

valid_mask = (
    all_meta["s1_idx"]
    .isin(valid_s1_ids)
)

X_train = X.loc[
    train_mask
].reset_index(
    drop=True
)

X_valid = X.loc[
    valid_mask
].reset_index(
    drop=True
)

y_train = y.loc[
    train_mask
].reset_index(
    drop=True
)

y_valid = y.loc[
    valid_mask
].reset_index(
    drop=True
)

valid_meta = all_meta.loc[
    valid_mask
].reset_index(
    drop=True
)

print(
    "Training rows:",
    f"{len(X_train):,}",
)

print(
    "Validation rows:",
    f"{len(X_valid):,}",
)

print(
    "Training positives:",
    f"{int(y_train.sum()):,}",
)

print(
    "Validation positives:",
    f"{int(y_valid.sum()):,}",
)

Training rows: 207,744
Validation rows: 51,408
Training positives: 15,481
Validation positives: 3,886


## 17. Train the Pairwise Classifier

Train an XGBoost binary classifier on the engineered candidate-pair features.

The classifier produces match probabilities; the final matching threshold is selected separately using entity-level macro F0.5.

In [30]:
model = XGBClassifier(
    **XGB_PARAMS
)

model.fit(
    X_train,
    y_train,
    eval_set=[
        (
            X_valid,
            y_valid,
        )
    ],
    verbose=False,
)

print(
    "XGBoost training complete."
)

XGBoost training complete.


In [35]:
valid_proba = model.predict_proba(
    X_valid
)[:, 1]

print(
    "Validation probabilities:",
    len(valid_proba),
)

print(
    "Probability range:",
    round(
        float(valid_proba.min()),
        4,
    ),
    "to",
    round(
        float(valid_proba.max()),
        4,
    ),
)

Validation probabilities: 51408
Probability range: 0.0 to 1.0


## 19. Define Entity-Level F0.5 Evaluation

Evaluate predictions independently for every source-1 entity.

The implementation gives a perfect score to correctly predicted empty entities and penalizes false matches for entities with no true matches.

In [31]:
def calculate_macro_f05(
    meta,
    probabilities,
    threshold,
    true_sets,
):

    predicted_sets = {}

    for i, row in enumerate(
        meta.itertuples(
            index=False
        )
    ):

        if (
            probabilities[i]
            >= threshold
        ):

            s1_idx = int(
                row.s1_idx
            )

            predicted_sets.setdefault(
                s1_idx,
                set(),
            ).add(
                (
                    row.source,
                    int(
                        row.matched_idx
                    ),
                )
            )

    scores = []

    for s1_idx, true_set in (
        true_sets.items()
    ):

        predicted_set = (
            predicted_sets.get(
                s1_idx,
                set(),
            )
        )

        if (
            not true_set
            and not predicted_set
        ):
            score = 1.0

        elif (
            not true_set
            and predicted_set
        ):
            score = 0.0

        elif (
            true_set
            and not predicted_set
        ):
            score = 0.0

        else:

            tp = len(
                true_set
                & predicted_set
            )

            fp = len(
                predicted_set
                - true_set
            )

            fn = len(
                true_set
                - predicted_set
            )

            precision = (
                tp / (tp + fp)
                if tp + fp > 0
                else 0.0
            )

            recall = (
                tp / (tp + fn)
                if tp + fn > 0
                else 0.0
            )

            if (
                precision == 0
                and recall == 0
            ):

                score = 0.0

            else:

                score = (
                    1.25
                    * precision
                    * recall
                ) / (
                    0.25 * precision
                    + recall
                )

        scores.append(score)

    return float(
        np.mean(scores)
    )

## 20. Prepare Validation Ground Truth

Create the true pair sets for validation source-1 entities.

Each true pair is represented by both its source and source-row index so source-2 and source-3 records remain distinct.

In [32]:
validation_s1_ids = (
    valid_meta[
        "s1_idx"
    ].unique()
)

valid_true_sets = {
    int(s1_idx): true_pairs_by_s1.get(
        int(s1_idx),
        set(),
    )
    for s1_idx in validation_s1_ids
}

print(
    "Validation S1 entities:",
    f"{len(valid_true_sets):,}",
)

print(
    "Validation entities with true matches:",
    sum(
        bool(v)
        for v in valid_true_sets.values()
    ),
)

Validation S1 entities: 1,840
Validation entities with true matches: 1752


## 21. Search for the Matching Threshold

Evaluate a range of probability thresholds against the entity-level macro F0.5 metric.

The selected threshold is optimized for the competition objective rather than ordinary classification accuracy.

In [36]:
thresholds = np.arange(
    0.10,
    0.991,
    0.02,
)

threshold_results = []

for threshold in thresholds:

    score = calculate_macro_f05(
        meta=valid_meta,
        probabilities=valid_proba,
        threshold=float(
            threshold
        ),
        true_sets=valid_true_sets,
    )

    threshold_results.append(
        {
            "threshold": float(
                threshold
            ),
            "macro_f05": float(
                score
            ),
        }
    )

threshold_results = pd.DataFrame(
    threshold_results
)

best_row = (
    threshold_results
    .loc[
        threshold_results[
            "macro_f05"
        ].idxmax()
    ]
)

BEST_THRESHOLD = float(
    best_row["threshold"]
)

BEST_F05 = float(
    best_row["macro_f05"]
)

print(
    "Best threshold:",
    round(
        BEST_THRESHOLD,
        4,
    ),
)

print(
    "Validation macro F0.5:",
    round(
        BEST_F05,
        5,
    ),
)

Best threshold: 0.4
Validation macro F0.5: 0.7471


## 22. Inspect Threshold Results

Display the strongest threshold candidates around the selected operating point.

This provides a compact view of how sensitive the validation score is to the final matching threshold.

In [37]:
threshold_results.sort_values(
    "macro_f05",
    ascending=False,
).head(10)

,threshold,macro_f05
15,0.40,0.747103
14,0.38,0.747073
13,0.36,0.747033
16,0.42,0.746518
12,0.34,0.746377
10,0.30,0.745893
31,0.72,0.745713
11,0.32,0.745695
30,0.70,0.745320
17,0.44,0.745023


## 23. Calculate Candidate-Level Metrics

Calculate ordinary precision, recall, and F0.5 on validation candidate pairs as supporting diagnostics.

These metrics are secondary because the competition evaluates macro F0.5 at the source-1 entity level.

In [40]:
valid_pred = (
    valid_proba
    >= BEST_THRESHOLD
).astype(
    np.int8
)

candidate_precision = (
    precision_score(
        y_valid,
        valid_pred,
        zero_division=0,
    )
)

candidate_recall = (
    recall_score(
        y_valid,
        valid_pred,
        zero_division=0,
    )
)

candidate_f05 = (
    fbeta_score(
        y_valid,
        valid_pred,
        beta=0.5,
        zero_division=0,
    )
)

print(
    "Candidate precision:",
    round(
        candidate_precision,
        5,
    ),
)

print(
    "Candidate recall:",
    round(
        candidate_recall,
        5,
    ),
)

print(
    "Candidate F0.5:",
    round(
        candidate_f05,
        5,
    ),
)

print(
    "Entity macro F0.5:",
    round(
        BEST_F05,
        5,
    ),
)

Candidate precision: 0.96102
Candidate recall: 0.96423
Candidate F0.5: 0.96166
Entity macro F0.5: 0.7471


## 24. Retrain the Final Matching Model

Retrain XGBoost using all sampled candidate pairs after the validation threshold has been selected.

The validation split is no longer needed because its purpose was only to select the operating threshold.

In [41]:
final_model = XGBClassifier(
    **XGB_PARAMS
)

final_model.fit(
    X,
    y,
    verbose=False,
)

print(
    "Final matching model trained."
)

Final matching model trained.


In [46]:
final_train_proba = final_model.predict_proba(
    X
)[:, 1]

print(
    "Final model training probabilities:",
    len(final_train_proba),
)

print(
    "Selected threshold:",
    round(
        BEST_THRESHOLD,
        4,
    ),
)

Final model training probabilities: 259152
Selected threshold: 0.4


## 25. Save the Matching Model

Save the final XGBoost model and selected probability threshold for use by Notebook 05.

Only the compact model artifacts are saved; the large preprocessed training tables are not written to disk.

In [48]:
MODEL_PATH = os.path.join(
    MODEL_DIR,
    "xgb_matching_model.json",
)

THRESHOLD_PATH = os.path.join(
    MODEL_DIR,
    "matching_threshold.txt",
)

final_model.save_model(
    MODEL_PATH
)

with open(
    THRESHOLD_PATH,
    "w",
) as f:

    f.write(
        f"{BEST_THRESHOLD:.8f}"
    )

print(
    "Model saved:",
    MODEL_PATH,
)

print(
    "Threshold saved:",
    THRESHOLD_PATH,
)

Model saved: /kaggle/working/matching_model/xgb_matching_model.json
Threshold saved: /kaggle/working/matching_model/matching_threshold.txt


## 26. Save Feature Metadata

Save the feature order used by the classifier so Notebook 05 can construct the inference matrix with exactly the same columns.

The metadata file contains only feature names and configuration values.

In [49]:
FEATURE_METADATA_PATH = os.path.join(
    MODEL_DIR,
    "feature_metadata.txt",
)

with open(
    FEATURE_METADATA_PATH,
    "w",
) as f:

    f.write(
        "FEATURE_COLUMNS\n"
    )

    for feature in FEATURE_COLUMNS:
        f.write(
            feature + "\n"
        )

    f.write(
        "\nTRAIN_BLOCKING_RULES\n"
    )

    for rule in TRAIN_BLOCKING_RULES:
        f.write(
            rule + "\n"
        )

print(
    "Feature metadata saved:",
    FEATURE_METADATA_PATH,
)

Feature metadata saved: /kaggle/working/matching_model/feature_metadata.txt


## 27. Final Model Summary

Summarize the trained model, number of training pairs, validation performance, and saved artifacts.

These values are the handoff information required by the prediction notebook.

In [50]:
print("=" * 60)
print("MATCHING MODEL COMPLETE")
print("=" * 60)

print(
    "Training S1 entities:",
    f"{len(sampled_s1_idx):,}",
)

print(
    "Training candidate pairs:",
    f"{len(X):,}",
)

print(
    "Positive pairs:",
    f"{int(y.sum()):,}",
)

print(
    "Negative pairs:",
    f"{int((y == 0).sum()):,}",
)

print(
    "Feature count:",
    len(FEATURE_COLUMNS),
)

print(
    "Validation macro F0.5:",
    round(
        BEST_F05,
        5,
    ),
)

print(
    "Selected threshold:",
    round(
        BEST_THRESHOLD,
        5,
    ),
)

print()
print(
    "Model:",
    MODEL_PATH,
)

print(
    "Threshold:",
    THRESHOLD_PATH,
)

print(
    "Metadata:",
    FEATURE_METADATA_PATH,
)

print("=" * 60)

MATCHING MODEL COMPLETE
Training S1 entities: 10,000
Training candidate pairs: 259,152
Positive pairs: 19,367
Negative pairs: 239,785
Feature count: 21
Validation macro F0.5: 0.7471
Selected threshold: 0.4

Model: /kaggle/working/matching_model/xgb_matching_model.json
Threshold: /kaggle/working/matching_model/matching_threshold.txt
Metadata: /kaggle/working/matching_model/feature_metadata.txt


In [51]:
import os
import shutil

MODEL_DIR = "/kaggle/working/matching_model"

OUTPUT_ZIP = "/kaggle/working/matching_model_v5"

shutil.make_archive(
    OUTPUT_ZIP,
    "zip",
    MODEL_DIR,
)

print(
    "Created:",
    OUTPUT_ZIP + ".zip",
)

print(
    "Files included:"
)

for filename in sorted(
    os.listdir(MODEL_DIR)
):
    print(
        " -",
        filename,
    )

Created: /kaggle/working/matching_model_v5.zip
Files included:
 - feature_metadata.txt
 - matching_threshold.txt
 - xgb_matching_model.json


In [52]:
import zipfile

with zipfile.ZipFile(
    "/kaggle/working/matching_model_v5.zip",
    "r",
) as z:

    print("\nZIP contents:")

    for name in z.namelist():
        print(" -", name)


ZIP contents:
 - matching_threshold.txt
 - xgb_matching_model.json
 - feature_metadata.txt


In [53]:
from IPython.display import FileLink, display

zip_path = "/kaggle/working/matching_model_v5.zip"

display(
    FileLink(
        zip_path,
        result_html_prefix="DOWNLOAD MODEL FILE: "
    )
)

/kaggle/working/matching_model_v5.zip

In [54]:
import base64
from IPython.display import HTML

def create_download_link(filename, title="Click here to download: "):
    # Reads the file binary data and encodes it to text
    with open(filename, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    
    # Generates a direct embedded download anchor
    html = f'<a download="{filename}" href="data:application/zip;base64,{b64}" target="_blank" style="color: #007bff; font-weight: bold; text-decoration: underline;">{title} {filename}</a>'
    return HTML(html)

# Generate the forced download link
create_download_link("matching_model_v5.zip")
